# # **LESSON 2: WHEN SET THEORY FALLS INTO SQL**
## *The Database Disaster of Unions, Intersections and Other Structured Nonsense*

> [!WARNING]
> ### **WARNING!**
> This lesson contains:
> - SQL,
> - set theory,
> - suspicious vegetables,
> - and the horrifying discovery that databases can do maths in a tie.
>

---

## **CHAPTER TWO: THE REVENGE OF THE TABLES**

You thought set theory lived safely in maths books.

You fool.

It has escaped.

Now it lurks in databases, hiding inside neat little tables, waiting for an innocent student to type something like:

In [11]:
SELECT * FROM Things;

(Command completed successfully)


and accidentally summon **logic**.

Because SQL is full of set theory.

Not the dramatic sort with thunder and screaming.

The boring-looking sort.

Which is much worse.

In set theory, you have collections of things.

In SQL, you have **result sets**.

That means every time you run a query, SQL hands you back a little bundle of rows and says:

> “Here. A set. Try not to lick it.”

So if set theory is about collecting things together, SQL is what happens when someone gives those collections ID numbers and stores them in a machine.

---

## **TODAY'S HORRIBLE PLAN**

Today you will see:

- what a **union** looks like in SQL,
- what an **intersection** looks like in SQL,
- how the SQL syntax matches the maths idea,
- and why this is actually much less scary than it sounds.

---

## **THE BASIC IDEA**

In set theory:

- a **union** means: everything in set \(A\), everything in set \(B\), all together
- an **intersection** means: only the things that are in **both** \(A\) and \(B\)

Mathematically, that looks like this.

### **Union**
$$
A \cup B
$$

### **Intersection**
$$
A \cap B
$$

Now here comes the alarming bit:

SQL does this too.

---

## **SQL'S UNION: THE GIANT MASH-UP**

Suppose one query gives you a set of rows.

And another query gives you another set of rows.

If you use:

```sql
UNION
```

SQL glues them together into one bigger result.

That is the database version of:

$$
A \cup B
$$

### **BUT BE CAREFUL!**
`UNION` removes duplicates.

That means if the same row appears in both results, SQL only shows it once.

So `UNION` is tidy, strict and slightly smug.

---

> [!NOTE]
> ### **HORRIBLE FACT**
> SQL also has:
>
> ```sql
> UNION ALL
> ```
>
> which keeps duplicates.
>
> In other words:
> - `UNION` says: *“One of each, please.”*
> - `UNION ALL` says: *“Throw them all in. I have lost control.”*

---

## **SQL'S INTERSECT: THE SNOOTY OVERLAP**

If you use:

```sql
INTERSECT
```

SQL gives you only the rows that appear in **both** queries.

That is the database version of:

$$
A \cap B
$$

So if one query finds:

- all customers who like olives

and another finds:

- all customers who like capers

then `INTERSECT` gives you:

- customers who like **both**

Which is either useful filtering or a very odd dinner party.

---

## **THE IMPORTANT RULE THAT SQL DEMANDS LIKE A TYRANT**

For `UNION`, `UNION ALL`, and `INTERSECT` to work nicely:

- both queries should return the **same number of columns**
- the columns should be in the **same order**
- the data types should be compatible

So this is good:

In [24]:
SELECT w.[Name] FROM dbo.Wizards w
UNION
SELECT v.[Name] FROM Vegetables v;

(16 rows affected)

Name        
------------
Beetroot    
Broccoli    
Bumblefog   
Carrot      
Cucumber    
Fizzlebark  
Grumblewand 
Muffinspell 
Parsnip     
Snortleflame
Spinach     
Toadwink    
Turnip      
Wizzlebeard 
Zapples     
Zucchini    
(16 rows)


It is deeply strange, but technically allowed.

This is bad:

In [31]:
SELECT w.[Name], w.Age FROM dbo.Wizards w
UNION
SELECT v.[Name] FROM dbo.Vegetables v;

(Command completed successfully)

That makes SQL cross its arms and glare at you.

---

## **A TINY, HORRIBLE EXAMPLE**

Imagine you have one query that returns:

```text
Sock
Worm
Olive
```

and another that returns:

```text
Olive
Caper
Worm
```

Then:

### **UNION**
gives you everything from both, without duplicates:

```text
Sock
Worm
Olive
Caper
```

### **INTERSECT**
gives you only what appears in both:

```text
Worm
Olive
```

Set theory is nodding proudly in the background.

---

## **THE SQL VERSION OF BLOBBY DIAGRAMS**

In maths, people draw circles.

In SQL, people write queries.

Same idea. Worse outfit.

If:

- query \(A\) returns one set of rows
- query \(B\) returns another set of rows

then:

### **Union**
$$
A \cup B
$$

becomes:

```sql
QueryA
UNION
QueryB
```

### **Intersection**
$$
A \cap B
$$

becomes:

```sql
QueryA
INTERSECT
QueryB
```

So SQL is basically doing set theory in office clothes.

---

> [!TIP]
> ### **SURVIVAL RULE**
> If you forget what they do:
>
> - `UNION` = **everything from both**
> - `INTERSECT` = **only shared rows**
>
> If in doubt, imagine two disgusting lunchboxes.
>
> `UNION` empties both lunchboxes onto the same tray.
>
> `INTERSECT` keeps only the identical horrors found in both.

---

## **THE PRACTICAL EXPERIMENT OF DOOM**

Let’s pretend you are running a tiny snack database.

You have two tables:

- one table of people who like **olives**
- one table of people who like **capers**

Now you want to know:

1. who likes olives **or** capers?
2. who likes olives **and** capers?

This is where SQL stops being a filing cabinet and becomes a maths goblin.

---
## **QUERY 1: THE UNION**

In [18]:
SELECT p.PersonName
FROM LikesOlives lo
JOIN People p
    ON p.PersonId = lo.PersonId

UNION

SELECT p.PersonName
FROM LikesCapers lc
JOIN People p
    ON p.PersonId = lc.PersonId;

(7 rows affected)

PersonName
----------
Asha      
Bilal     
Cerys     
Dilan     
Elif      
Farah     
Goran     
(7 rows)

### **What this means**
Give me everybody from both tables.

No duplicates.

If someone appears in both, show them only once.

That is:

$$
A \cup B
$$

---

## **QUERY 2: THE INTERSECTION**

In [19]:
SELECT p.PersonName
FROM LikesOlives lo
JOIN People p
    ON p.PersonId = lo.PersonId

INTERSECT

SELECT p.PersonName
FROM LikesCapers lc
JOIN People p
    ON p.PersonId = lc.PersonId;

(3 rows affected)

PersonName
----------
Bilal     
Dilan     
Goran     
(3 rows)

## **WHY NOT JUST USE JOINS?**

Excellent question, O Suspicious Student.

You *can* often use `JOIN`s to find overlaps.

But `INTERSECT` is nice when you want to say:

> “Give me the rows returned by both of these queries.”

It expresses the set idea very directly.

A join is more like:

> “Match rows from these tables according to some relationship.”

An `INTERSECT` is more like:

> “These two result sets. Which rows are shared?”

It is elegant.

In a chilly, mathematical sort of way.

## **THE DIFFERENCE BETWEEN UNION AND UNION ALL**

### `UNION`
Removes duplicates.

### `UNION ALL`
Keeps duplicates.

So if `Bilal` appears in both result sets:

- `UNION` shows `Bilal` once
- `UNION ALL` shows `Bilal` twice

Which means `UNION ALL` is less tidy, but sometimes faster and exactly what you want.

In [ ]:
select p.PersonName from LikesOlives lo join People p on p.PersonId = lo.PersonId
union
select p.PersonName from LikesCapers lc join People p on p.PersonId = lc.PersonId;

select p.PersonName from LikesOlives lo join People p on p.PersonId = lo.PersonId
INTERSECT
select p.PersonName from LikesCapers lc join People p on p.PersonId = lc.PersonId;

(5 rows affected)
(7 rows affected)

PersonName
----------
Asha      
Bilal     
Cerys     
Dilan     
Goran     
(5 rows)

PersonName
----------
Asha      
Bilal     
Cerys     
Dilan     
Elif      
Farah     
Goran     
(7 rows)

In [35]:
select p.PersonName from LikesOlives lo
JOIN People p
    ON p.PersonId = lo.PersonId
UNION ALL
select p.PersonName from LikesCapers lc
JOIN People p
    ON p.PersonId = lc.PersonId;

(10 rows affected)

PersonName
----------
Asha      
Bilal     
Cerys     
Dilan     
Goran     
Bilal     
Dilan     
Elif      
Farah     
Goran     
(10 rows)

In [ ]:
SELECT 
    COUNT(combined_names.PersonName) AS [name_count],
    combined_names.PersonName
FROM (
    SELECT p.PersonName
    FROM LikesOlives lo
    JOIN People p
        ON p.PersonId = lo.PersonId

    UNION ALL

    SELECT p.PersonName
    FROM LikesCapers lc
    JOIN People p
        ON p.PersonId = lc.PersonId
) AS combined_names
GROUP BY combined_names.PersonName
having COUNT(combined_names.PersonName) > 1
ORDER BY [name_count] DESC;

(7 rows affected)

name_count | PersonName
-----------+-----------
2          | Bilal     
2          | Dilan     
2          | Goran     
1          | Asha      
1          | Elif      
1          | Farah     
1          | Cerys     
(7 rows)

Like a messy backpack full of identical bananas.

---

## **THE NASTY LITTLE SUMMARY**

> ### **WHAT YOU HAVE LEARNED**
>
> In set theory:
>
> $$
> A \cup B
> $$
>
> means **union**: everything in either set.
>
> In SQL, that becomes:
>
> ```sql
> QueryA
> UNION
> QueryB
> ```
>
> In set theory:
>
> $$
> A \cap B
> $$
>
> means **intersection**: only what both sets share.
>
> In SQL, that becomes:
>
> ```sql
> QueryA
> INTERSECT
> QueryB
> ```
>
> Also:
>
> - `UNION` removes duplicates
> - `UNION ALL` keeps duplicates
> - `INTERSECT` keeps only shared rows

---

In [3]:
select * from fun.animals a where a.WeightKg >= 200

(56 rows affected)

AnimalId | Species                         | Name            | CommonName           | WeightKg  | HeightCm | LengthCm | CountryOfOriginId | DietId | RiskLevelId | EpochId | IsExtinct | DiscoveredOrCataloguedAt | LastSeenAt          | LifespanYears | HasWings | CanSwim | SpeedKph | FunFact                                         | Notes                                | CreatedAt          
---------+---------------------------------+-----------------+----------------------+-----------+----------+----------+-------------------+--------+-------------+---------+-----------+--------------------------+---------------------+---------------+----------+---------+----------+-------------------------------------------------+--------------------------------------+--------------------
1        | Tyrannosaurus rex               | SQL Rex         | Tyrannosaurus        | 8000.00   | 400.00   | 1200.00  | 5                 | 2      | 6           | 3       | 1         | 1902-01-01 00:00:00      | NULL  